# 🧹 Notebook 03 — Data Preprocessing Pipeline

**Project:** AI-Powered Demand Forecasting & Inventory Optimization  
**Author:** NTI Capstone Team  
**Environment:** Local / Google Colab

---

## 🎯 Purpose & Scope

Clean, harmonize, downcast, filter, and merge all raw datasets into a single  
**memory-optimized, production-ready DataFrame** saved as `clean_data.parquet`.

### ⚡ RAM Optimization Strategy

The raw `train.parquet` has **125.5 million rows**. We apply aggressive memory optimizations  
and a **2015-01-01+ Temporal Filter** (drops 38.6M noisy 2013–2014 rows), reducing memory from 15 GB to **~2.2 GB**.

| Optimization | Before | After | Savings |
|-------------|--------|-------|--------|
| Drop `id` column | int64 (1 GB) | Removed | 100% |
| `dcoilwtico` | float64 (1 GB) | float32 (0.5 GB) | 50% |
| `class` | int64 (1 GB) | int16 (0.25 GB) | 75% |
| `holiday_type` | object (1+ GB) | category (0.13 GB) | 87% |
| `transactions` | int32 (0.5 GB) | int16 (0.25 GB) | 50% |
| **Temporal Filter (`2015-01-01+`)** | 125.5M rows | **86.9M rows** | **~31% row reduction** |
| **Total** | **~15 GB** | **~2.2 GB** | **~85% RAM Savings** |

| Specification | Detail |
|---------------|--------|
| **Inputs** | 6 Parquet datasets from `01_Dataset/parquet/` |
| **Output** | `01_Dataset/processed/clean_data.parquet` (86.9M rows, ~2.2 GB) |
| **Previous** | `02_eda.ipynb` |
| **Next** | `04_feature_engineering.ipynb` |

---


## 1️⃣ Environment Setup & Project Bootstrap


In [1]:
# ============================================================
# Project Bootstrap (Google Colab + Local)
# Run this cell first in every notebook.
# ============================================================
import os, sys
from pathlib import Path

# 1. Mount Google Drive (Colab only)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# 2. Find project root (checks Drive paths + local paths)
POSSIBLE_ROOTS = [
    # Google Drive paths
    Path('/content/drive/MyDrive/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/NTI/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/Colab Notebooks/Demand-Forecasting-System'),
    # Local paths (for VS Code / Jupyter)
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in POSSIBLE_ROOTS:
    if p.exists() and (p / 'config.py').exists():
        PROJECT_ROOT = p.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '❌ Project root with config.py not found.\n'
        'Colab: Make sure the folder is in your Google Drive (MyDrive/Demand-Forecasting-System).\n'
        'Shared with me? Right-click → Organize → Add shortcut to My Drive.\n'
        'Local: Run the notebook from inside the project directory.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

ENV = 'Google Colab' if 'google.colab' in sys.modules else 'Local'
print('=' * 60)
print(f'📁 Project Root : {PROJECT_ROOT}')
print(f'📂 Working Dir  : {os.getcwd()}')
print(f'🖥️  Runtime      : {ENV}')
print('✅ Bootstrap OK')
print('=' * 60)


📁 Project Root : F:\NTI\Demand Forecasting System Backup
📂 Working Dir  : F:\NTI\Demand Forecasting System Backup
🖥️  Runtime      : Local
✅ Bootstrap OK


## 2️⃣ Imports & Configuration


In [2]:
import os, sys, time, gc, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import config, utils

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

OUTPUT_DIR = Path(config.PROJECT_ROOT) / 'output' / '03_preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Libraries Loaded!')


✅ Libraries Loaded!


## 3️⃣ Load Raw Datasets


In [3]:
df_train    = utils.load_train_parquet(verbose=True)
df_stores   = utils.load_parquet(config.STORES_PARQUET, verbose=True)
df_items    = utils.load_parquet(config.ITEMS_PARQUET, verbose=True)
df_oil      = utils.load_parquet(config.OIL_PARQUET, verbose=True)
df_trans    = utils.load_parquet(config.TRANSACTIONS_PARQUET, verbose=True)
df_holidays = utils.load_parquet(config.HOLIDAYS_PARQUET, verbose=True)
print('✅ All raw datasets loaded.')


Loading train.parquet...
Loaded successfully.
Rows: 125,497,040 | Columns: 6 | Memory usage: 14.89 GB | Elapsed time: 10.44s

Loading stores.parquet...
Loaded successfully.
Rows: 54 | Columns: 5 | Memory usage: 0.01 MB | Elapsed time: 0.03s

Loading items.parquet...
Loaded successfully.
Rows: 4,100 | Columns: 4 | Memory usage: 0.35 MB | Elapsed time: 0.35s

Loading oil.parquet...
Loaded successfully.
Rows: 1,218 | Columns: 2 | Memory usage: 0.06 MB | Elapsed time: 0.00s

Loading transactions.parquet...
Loaded successfully.
Rows: 83,488 | Columns: 3 | Memory usage: 4.46 MB | Elapsed time: 0.02s

Loading holidays_events.parquet...
Loaded successfully.
Rows: 350 | Columns: 6 | Memory usage: 0.10 MB | Elapsed time: 0.00s

✅ All raw datasets loaded.


## 4️⃣ Missing Value Imputation


In [4]:
# Impute oil price
df_oil['date'] = pd.to_datetime(df_oil['date'])
df_oil = df_oil.sort_values('date').reset_index(drop=True)
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].ffill().bfill()

# Impute promotion
df_train['onpromotion'] = df_train['onpromotion'].fillna(False).astype(bool)
print('✅ Imputation complete.')


✅ Imputation complete.


## 5️⃣ Negative Sales & Return Handling


In [5]:
df_train['is_return'] = (df_train['unit_sales'] < 0).astype('uint8')
df_train['unit_sales'] = df_train['unit_sales'].clip(lower=0)
print('✅ Negative sales clipped.')


✅ Negative sales clipped.


## 6️⃣ Date Parsing & Temporal Index


In [6]:
for df in [df_train, df_oil, df_trans, df_holidays]:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
print('✅ Date columns parsed to datetime64[ns].')


✅ Date columns parsed to datetime64[ns].


## 7️⃣ Memory Downcasting & `id` Column Removal


In [7]:
if 'id' in df_train.columns:
    df_train.drop(columns=['id'], inplace=True)

df_train['store_nbr']  = df_train['store_nbr'].astype('uint8')
df_train['item_nbr']   = df_train['item_nbr'].astype('uint32')
df_train['unit_sales'] = df_train['unit_sales'].astype('float32')

df_stores['store_nbr'] = df_stores['store_nbr'].astype('uint8')
df_stores['cluster']   = df_stores['cluster'].astype('uint8')
for col in ['city', 'state', 'type']:
    df_stores[col] = df_stores[col].astype('category')

df_items['item_nbr']   = df_items['item_nbr'].astype('uint32')
df_items['perishable'] = df_items['perishable'].astype('uint8')
if 'class' in df_items.columns:
    df_items['class'] = df_items['class'].astype('int16')
for col in ['family']:
    if col in df_items.columns:
        df_items[col] = df_items[col].astype('category')

df_oil['dcoilwtico'] = df_oil['dcoilwtico'].astype('float32')
gc.collect()
print('✅ Downcasting complete.')


✅ Downcasting complete.


## 8️⃣ Duplicate Detection


In [8]:
for name, (df, pk) in {'train': (df_train, ['date', 'store_nbr', 'item_nbr'])}.items():
    dupes = df.duplicated(subset=pk).sum()
    if dupes > 0:
        df.drop_duplicates(subset=pk, keep='first', inplace=True)
print('✅ Primary key check complete.')


✅ Primary key check complete.


## 9️⃣ Holiday Calendar Processing


In [9]:
df_holidays_clean = df_holidays[~((df_holidays['transferred'] == True) & (df_holidays['type'] != 'Transfer'))].copy()
holiday_dates = df_holidays_clean.groupby('date').agg(is_holiday=('type', 'count'), holiday_type=('type', 'first')).reset_index()
holiday_dates['is_holiday'] = 1
print('✅ Holidays processed.')


✅ Holidays processed.


## 🔟 Oil Price Time-Series Completion


In [10]:
train_date_range = pd.date_range(start=df_train['date'].min(), end=df_train['date'].max(), freq='D')
oil_complete = pd.DataFrame({'date': train_date_range}).merge(df_oil, on='date', how='left')
oil_complete['dcoilwtico'] = oil_complete['dcoilwtico'].ffill().bfill().astype('float32')
print('✅ Oil series completed.')


✅ Oil series completed.


## 1️⃣1️⃣ Relational Merging Pipeline


In [11]:
df_train = df_train.merge(df_stores, on='store_nbr', how='left')
df_train = df_train.merge(df_items, on='item_nbr', how='left')
df_train = df_train.merge(oil_complete[['date', 'dcoilwtico']], on='date', how='left')
df_train = df_train.merge(df_trans, on=['date', 'store_nbr'], how='left')
df_train = df_train.merge(holiday_dates[['date', 'is_holiday', 'holiday_type']], on='date', how='left')

df_train['is_holiday']   = df_train['is_holiday'].fillna(0).astype('uint8')
df_train['holiday_type'] = df_train['holiday_type'].fillna('Regular').astype('category')
df_train['transactions'] = df_train['transactions'].fillna(0).astype('int16')
print('✅ Merges complete.')


✅ Merges complete.


## 1️⃣2️⃣ Final Type Enforcement — Zero Object Columns


In [12]:
for col in df_train.columns:
    dt = str(df_train[col].dtype)
    if dt == 'object' or dt == 'string':
        df_train[col] = df_train[col].astype('category')
    elif dt == 'float64':
        df_train[col] = df_train[col].astype('float32')
print('✅ Zero object columns enforced.')


✅ Zero object columns enforced.


## 1️⃣3️⃣ Temporal Filtering (`2015-01-01+`) for RAM Optimization

Filters out 2013–2014 data (~38.6M noisy rows) to keep a clean, high-signal 86.9M row dataset.


In [13]:
# ── Temporal Filter ──
START_DATE = '2015-01-01'
rows_before = len(df_train)
df_train = df_train[df_train['date'] >= START_DATE].reset_index(drop=True)
print(f'✂️ Filtered date range: >= {START_DATE}')
print(f'   Rows BEFORE: {rows_before:,}')
print(f'   Rows AFTER:  {len(df_train):,} ({(1 - len(df_train)/rows_before)*100:.1f}% reduction)')
print(f'📊 Final Memory Footprint: {df_train.memory_usage(deep=True).sum() / 1e9:.2f} GB')
gc.collect()


✂️ Filtered date range: >= 2015-01-01
   Rows BEFORE: 125,497,040
   Rows AFTER:  86,902,776 (30.8% reduction)
📊 Final Memory Footprint: 3.04 GB


0

## 1️⃣4️⃣ Export Processed Dataset (`clean_data.parquet`)


In [14]:
output_dir = Path(config.PROJECT_ROOT) / '01_Dataset' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'clean_data.parquet'

t0 = time.time()
df_train.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)
print(f'🎉 clean_data.parquet exported!')
print(f'   Rows: {len(df_train):,} | Columns: {len(df_train.columns)}')
print(f'   File size: {output_path.stat().st_size / 1e9:.2f} GB | Write time: {time.time()-t0:.1f}s')


🎉 clean_data.parquet exported!
   Rows: 86,902,776 | Columns: 17
   File size: 0.40 GB | Write time: 23.1s
